In [3]:
# Environment Sync & Directory Alignment
from google.colab import drive
import os
import sys

drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/adaptive_crypto_framework'
os.chdir(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# KNN Training, Hyperparameter Optimization, and Metric Graphing
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Load engineered dataset from Notebook 1
dataset = pd.read_csv('data/processed/sensitivity_dataset.csv')
X = dataset[['Ext_ID', 'Size_KB', 'Entropy', 'Keywords']]
y = dataset['Is_Sensitive']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Grid search for the best K-Neighbor coordinate
best_k = 1
best_score = 0
for k in [1, 3, 5, 7]:
    test_knn = KNeighborsClassifier(n_neighbors=k)
    test_knn.fit(X_train, y_train)
    score = accuracy_score(y_test, test_knn.predict(X_test))
    if score > best_score:
        best_score, best_k = score, k

# Deploy final model
optimized_knn = KNeighborsClassifier(n_neighbors=best_k)
optimized_knn.fit(X_train, y_train)

# Save the model object to disk for deployment
with open('src/knn_model.pkl', 'wb') as f:
    pickle.dump(optimized_knn, f)

print(f"🎯 Model optimized and saved using K={best_k} neighbors!")
print(classification_report(y_test, optimized_knn.predict(X_test), target_names=['Normal (0)', 'Sensitive (1)']))

🎯 Model optimized and saved using K=1 neighbors!
               precision    recall  f1-score   support

   Normal (0)       1.00      1.00      1.00       118
Sensitive (1)       1.00      1.00      1.00       122

     accuracy                           1.00       240
    macro avg       1.00      1.00      1.00       240
 weighted avg       1.00      1.00      1.00       240

